In [ ]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/SK_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Mounted at /content/drive


In [ ]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000000000_0 -0.248135       1  0.590481  0.284002  0.794735   
1  00000000000000000000_0 -0.241654       1  0.546826  0.262380  0.672039   
2  00000000000000000000_0 -0.161485       1  0.415112  0.213809  0.552272   
3  00000000000000000000_0 -0.144725       1  0.394572  0.189810  0.540123   
4  00000000000000000000_0 -0.146454       1  0.393314  0.195709  0.551384   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  88.136857   
1  2021  {"geodesic":false,"type":"Point","coordinates"...  88.136857   
2  2022  {"geodesic":false,"type":"Point","coordinates"...  88.136857   
3  2022  {"geodesic":false,"type":"Point","coordinates"...  88.136857   
4  2023  {"geodesic":false,"type":"Point","coordinates"...  88.136857   

    latitude  
0  27.116994  
1  27.116994  
2  27.116994  
3  27.116994  
4  27.116994  
[2021 20

In [ ]:
# =====================================================
# 1. Imports
# =====================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import classification_report

# =====================================================
# 2. Create Labels (NDVI drop & BSI rise)
# =====================================================
# X_raw shape: (samples, time_steps, features)
# locations shape: (samples, 2) -> latitude, longitude

ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # NDVI 2021-2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # BSI 2024-2021

# Use 85th percentile thresholds (adjust if too small)
ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# =====================================================
# 3. Train-Test Split (Safe for small datasets)
# =====================================================
min_class_count = np.min([np.sum(y==0), np.sum(y==1)])
if min_class_count < 2:
    print("WARNING: Tiny dataset, skipping stratified split.")
    X_train, X_test, y_train, y_test, loc_train, loc_test = (
        X_raw, X_raw, y, y, locations, locations
    )
else:
    X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
        X_raw, y, locations,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# =====================================================
# 4. Scaling (No data leakage)
# =====================================================
scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# =====================================================
# 5. LSTM Model
# =====================================================
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Recall', 'Precision']
)

model.summary()

# =====================================================
# 6. Train Model (Use class weighting if enough samples)
# =====================================================
if min_class_count < 2:
    print("Skipping class weighting due to tiny dataset.")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=1,  # small batch for tiny data
        verbose=1
    )
else:
    class_weight = {0: 1, 1: 12}
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=128,
        class_weight=class_weight,
        verbose=1
    )

# =====================================================
# 7. Evaluation
# =====================================================
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)  # recall-friendly threshold

print(classification_report(y_test, y_pred))

# =====================================================
# 8. Save Predictions (with locations)
# =====================================================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/SK_Deforestation_Predictions.csv',
    index=False
)

print("✅ Saved: SK_Deforestation_Predictions.csv")


Label distribution:
No deforestation: 4
Deforestation: 1
Train shape: (5, 4, 5)
Test shape : (5, 4, 5)
Scaled train shape: (5, 4, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Skipping class weighting due to tiny dataset.
Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 215ms/step - Precision: 0.4139 - Recall: 1.0000 - accuracy: 0.4139 - loss: 0.7057 - val_Precision: 0.3333 - val_Recall: 1.0000 - val_accuracy: 0.6000 - val_loss: 0.6942
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.1750 - loss: 0.7028 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.8000 - val_loss: 0.6719
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.8917 - loss: 0.6715 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.8000 - val_loss: 0.6528
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.7528 - loss: 0.6581 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.8000 - val_loss: 0.6372
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - Precision: 0.0000e+00 - Re

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/SK_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/SK_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary (Sikkim):")
print(df_merged['cause'].value_counts())

NDVI: {'mean': np.float64(-0.059487531999999996), 'std': 0.1960504197641425, 'q1': np.float64(-0.26335563000000006), 'q3': np.float64(0.06336697000000002), 'lower_2std': np.float64(-0.451588371528285), 'upper_2std': np.float64(0.332613307528285)}
NBR : {'mean': np.float64(-0.12673530400000002), 'std': 0.09379432477576868, 'q1': np.float64(-0.18013015), 'q3': np.float64(-0.03156305000000004), 'lower_2std': np.float64(-0.3143239535515374), 'upper_2std': np.float64(0.06085334555153735)}
BSI : {'mean': np.float64(0.04447436351843118), 'std': 0.05887295373530091, 'q1': np.float64(-0.013515427434107607), 'q3': np.float64(0.0885925791411141), 'lower_2std': np.float64(-0.07327154395217064), 'upper_2std': np.float64(0.162220270989033)}
NDMI: {'mean': np.float64(-0.056468125999999993), 'std': 0.05499671307358586, 'q1': np.float64(-0.07373896999999999), 'q3': np.float64(-0.00614845), 'lower_2std': np.float64(-0.16646155214717173), 'upper_2std': np.float64(0.05352530014717173)}

Deforestation Caus

In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Sikkim map
# SK roughly: latitude 27.0–28.0, longitude 88.0–88.9
m = folium.Map(location=[27.5, 88.5], zoom_start=8)

# Load CSV (Sikkim adaptive causes)
df = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())

cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)

# Save map
m.save('/content/drive/MyDrive/SK_Deforestation_Map.html')
print("✅ Sikkim deforestation map saved successfully!")

Series([], Name: count, dtype: int64)
✅ Sikkim deforestation map saved successfully!


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Predictions.csv')
print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Causes_Adaptive.csv')
print(df_cause.head())

# ==============================
# ENSURE LAT/LON TYPES MATCH
# ==============================
for col in ['latitude', 'longitude']:
    df_deforest[col] = df_deforest[col].astype(float)
    df_cause[col] = df_cause[col].astype(float)

# ==============================
# MERGE CAUSE DATA
# ==============================
if len(df_deforest) > 0:
    df_deforest = df_deforest.merge(
        df_cause[['latitude', 'longitude', 'cause']],
        on=['latitude', 'longitude'],
        how='left'
    )
else:
    # If no deforestation samples, create empty 'cause' column
    df_deforest['cause'] = pd.Series(dtype=str)

print(df_deforest.head())

# ==============================
# LOAD SIKKIM DISTRICTS
# ==============================
sk = gpd.read_file('/content/drive/MyDrive/SK.geojson')

# CRS SAFETY
if sk.crs is None:
    sk.set_crs("EPSG:4326", inplace=True)
sk = sk.to_crs("EPSG:4326")
sk["state"] = "Sikkim"

print(sk.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
if len(df_deforest) > 0:
    geometry = [Point(xy) for xy in zip(df_deforest['longitude'], df_deforest['latitude'])]
    gdf_points = gpd.GeoDataFrame(df_deforest, geometry=geometry, crs="EPSG:4326")
else:
    # Create empty GeoDataFrame with same columns
    gdf_points = gpd.GeoDataFrame(columns=list(df_deforest.columns) + ['geometry'], crs="EPSG:4326")

print(gdf_points.head())

Total samples: 5
    latitude  longitude  deforestation
0  27.116994  88.136857              0
1  27.116994  88.137755              0
2  27.929071  88.538404              0
3  27.929970  88.538404              0
4  27.931766  88.543793              0
Total deforestation samples: 0
Empty DataFrame
Columns: [latitude, longitude, deforestation, NDVI_change, NBR_change, BSI_change, NDMI_change, cause]
Index: []
Empty DataFrame
Columns: [latitude, longitude, deforestation, cause]
Index: []
  REMARKS_2 Country State_Name State_Code        Dist_Name Dist_Code  \
0      None   India     Sikkim         11    East District       244   
1      None   India     Sikkim         11  North  District       241   
2      None   India     Sikkim         11   South District       243   
3      None   India     Sikkim         11    West District       242   

                                            geometry   state  
0  POLYGON ((88.80601 27.41424, 88.80712 27.4112,...  Sikkim  
1  POLYGON ((88.64526 2

In [ ]:
# =====================================================
# SIKKIM FOREST MAP (Deforestation + Causes)
# =====================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import MarkerCluster
import numpy as np

# =====================================================
# STEP 1: Load Sikkim districts
# =====================================================
sk = gpd.read_file('/content/drive/MyDrive/SK.geojson')

if sk.crs is None:
    sk.set_crs("EPSG:4326", inplace=True)
sk = sk.to_crs("EPSG:4326")
sk["state"] = "Sikkim"
gdf_districts = sk.copy()

# =====================================================
# STEP 2: Load deforestation predictions
# =====================================================
df_def = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Predictions.csv')

# Ensure afforestation column exists
if 'afforestation' not in df_def.columns:
    df_def['afforestation'] = 0

# =====================================================
# STEP 3: Load cause data
# =====================================================
df_cause = pd.read_csv('/content/drive/MyDrive/SK_Deforestation_Causes_Adaptive.csv')

# Merge causes safely (handle float/object mismatch)
df_def['latitude'] = df_def['latitude'].astype(float)
df_def['longitude'] = df_def['longitude'].astype(float)
df_cause['latitude'] = df_cause['latitude'].astype(float)
df_cause['longitude'] = df_cause['longitude'].astype(float)

df_def = pd.merge(
    df_def,
    df_cause[['latitude','longitude','cause']],
    on=['latitude','longitude'],
    how='left'
)

# Fill missing causes
df_def['cause'].fillna('Unknown', inplace=True)

# =====================================================
# STEP 4: Convert predictions to GeoDataFrame
# =====================================================
geometry = [Point(xy) for xy in zip(df_def['longitude'], df_def['latitude'])]
gdf_points = gpd.GeoDataFrame(df_def, geometry=geometry, crs="EPSG:4326")

# =====================================================
# STEP 5: Spatial join with districts
# =====================================================
points_with_district = gpd.sjoin(gdf_points, gdf_districts, how='left', predicate='within')

# =====================================================
# STEP 6: District statistics
# =====================================================
district_total = points_with_district.groupby('Dist_Name').size().reset_index(name='total_samples')
district_deforestation = points_with_district[points_with_district['deforestation']==1] \
    .groupby('Dist_Name').size().reset_index(name='deforestation_samples')
district_afforestation = points_with_district[points_with_district['afforestation']==1] \
    .groupby('Dist_Name').size().reset_index(name='afforestation_samples')

# Merge stats into district GeoDataFrame
gdf_districts = gdf_districts.merge(district_total, on='Dist_Name', how='left') \
                             .merge(district_deforestation, on='Dist_Name', how='left') \
                             .merge(district_afforestation, on='Dist_Name', how='left')

gdf_districts.fillna(0, inplace=True)

# Area & rate calculations
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (gdf_districts["deforestation_samples"]/gdf_districts["total_samples"]*100).fillna(0)
gdf_districts["deforestation_area_sqkm"] = gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
gdf_districts["deforestation_area_ha"] = gdf_districts["deforestation_samples"] * PIXEL_AREA_HA

gdf_districts["afforestation_rate_%"] = (gdf_districts["afforestation_samples"]/gdf_districts["total_samples"]*100).fillna(0)
gdf_districts["afforestation_area_sqkm"] = gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
gdf_districts["afforestation_area_ha"] = gdf_districts["afforestation_samples"] * PIXEL_AREA_HA

# =====================================================
# STEP 7: Create Folium map (center Sikkim)
# =====================================================
m = folium.Map(location=[27.5, 88.5], zoom_start=8)

# =====================================================
# STEP 8: Add choropleth (handle uniform data)
# =====================================================
unique_vals = gdf_districts["deforestation_samples"].nunique()

if unique_vals < 3:
    # Use single color if all zeros
    folium.GeoJson(
        gdf_districts,
        style_function=lambda feature: {
            'fillColor': 'orange',
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['Dist_Name','deforestation_samples'],
            aliases=['District:','Deforestation samples:'],
            localize=True
        )
    ).add_to(m)
else:
    min_val = gdf_districts["deforestation_samples"].min()
    max_val = gdf_districts["deforestation_samples"].max()
    bins = np.linspace(min_val, max_val, num=6).tolist()

    folium.Choropleth(
        geo_data=gdf_districts.to_json(),
        data=gdf_districts,
        columns=["Dist_Name","deforestation_samples"],
        key_on="feature.properties.Dist_Name",
        fill_color="YlOrRd",
        bins=bins,
        fill_opacity=0.7,
        line_opacity=0.4,
        legend_name="Deforestation Samples (Sikkim)"
    ).add_to(m)

# =====================================================
# STEP 9: Add MarkerCluster for causes
# =====================================================
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray',
    'Unknown': 'blue'
}

marker_cluster = MarkerCluster().add_to(m)

for _, row in gdf_points.iterrows():
    color = cause_colors.get(row['cause'], 'blue')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7
    ).add_to(marker_cluster)

# =====================================================
# STEP 10: Alert Popup (top districts)
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = gdf_districts.sort_values(by="deforestation_samples", ascending=False).head(3)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Sikkim Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[27.5, 88.5],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 11: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/SK_forest.html")
print("✅ Sikkim map saved successfully!")

✅ Sikkim map saved successfully!


/tmp/ipykernel_217/765701606.py:51: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_def['cause'].fillna('Unknown', inplace=True)
/tmp/ipykernel_217/765701606.py:78: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)
